PII Middleware


In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_groq import ChatGroq
from langchain_core.tools import tool

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [ ]:
@tool
def simple_tool(msg: str)-> str:
    """ look up customer information."""
    return f"Customer record found for query ;{msg}"


In [ ]:
# create agent with PII middlewaer
agent=create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=[simple_tool],
    middleware=[
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),
        PIIMiddleware(
                    "credit_card",
                    strategy="mask",
                    apply_to_input=True,
        ),
        PIIMiddleware(
                    "api_key",
                    detector=r"gsk_[a-zA-Z)-9]{32}",
                    strategy="block",
                    apply_to_input=True,
        ),

    ]
)

In [ ]:
res=agent.invoke({"messages":[{"role": "user", "content":"hey I am john my email id is can you help me"}]})
res

In [ ]:
res["messages"][-1].content

In [ ]:
try:
    res=agent.invoke({"messages":[{"role": "user", "content":". is  can you help me"}]})
    res
except Exception as e:
    print("blocked as expected")

Strategies:
### Strategies

| Strategy | Result |
|---|---|
| `redact` | `[REDACTED_EMAIL]` |
| `mask` | `****-****-****-1234` |
| `hash` | `a8f5f167...` |
| `block` | Raises an exception |

HUMAN IN THE LOOP

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain.agents.middleware import HumanInTheLoopMiddleware

In [ ]:
from langchain_tavily import TavilySearch
search_tool=TavilySearch(max_results=2)

# customer tools when human needed
@tool
def send_email(msg:str)-> str:
    """ this tool will send email"""
    return f"mail has been sent"

@tool
def delete_data(msg:str)-> str:
    """ this tool will delete data """
    return f"data deleted"

In [ ]:
agent=create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=[search_tool,send_email,delete_data],
    middleware=[
        HumanInTheLoopMiddleware(
                interrupt_on={
                    "search_tool": False,
                    "delete_data":True,
                    "send_email":True,
                }
            ),
    ],
    checkpointer=InMemorySaver(),
)


In [ ]:
config1= {"configurable":{"thread_id":"user1234"}}
res=agent.invoke(
    {"messages":[{"role":"user", "content":"give recent ai news in max 5 lines"}]},
    config=config1
    )
res

In [ ]:
from IPython.display import display,Markdown
display(Markdown(res["messages"][-1].content))

human

In [ ]:
res=agent.invoke(
    {"messages":[{"role":"user", "content":"this is google ceo email suder@google.com send I am fine msg"}]},
    config=config1
    )
res

In [ ]:
#human approving 
approved_result=agent.invoke(
    Command(resume={"decisions":[{"type": "approve"}]}),
    config=config1
)
approved_result

In [ ]:
display(Markdown(approved_result["messages"][-1].content))

In [ ]:
import ipywidgets as widgets
from IPython.display import display
from langgraph.types import Command

config = {
    "configurable": {
        "thread_id": "email-session-1"
    }
}

def show_approval(result):
    interrupts = result.get("__interrupt__", [])

    if not interrupts:
        print("No approval required.")
        return

    approve_btn = widgets.Button(
        description="Approve",
        button_style="success",
        icon="check"
    )

    reject_btn = widgets.Button(
        description="Reject",
        button_style="danger",
        icon="times"
    )

    status = widgets.Output()

    def handle_decision(decision):
        approve_btn.disabled = True
        reject_btn.disabled = True

        with status:
            print(f"Decision: {decision}")

            response = agent.invoke(
                Command(
                    resume={
                        "decisions": [
                            {"type": decision}
                        ]
                    }
                ),
                config=config
            )

            print("Agent resumed.")
            print(response)

    approve_btn.on_click(
        lambda _: handle_decision("approve")
    )

    reject_btn.on_click(
        lambda _: handle_decision("reject")
    )

    display(
        widgets.VBox([
            widgets.HTML(
                "<h3>Email approval required</h3>"
                "<p>Review the proposed email before proceeding.</p>"
            ),
            widgets.HBox([
                approve_btn,
                reject_btn
            ]),
            status
        ])
    )

In [ ]:
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "this is google ceo email suder@google.com send I am fine msg"
            }
        ]
    },
    config=config
)
result


In [ ]:
show_approval(result)

In [ ]:
result

In [ ]:
import ipywidgets as widgets
from IPython.display import display

display(widgets.Button(
    description="Approve",
    button_style="success"
))

In [ ]:
import ipywidgets as widgets
from IPython.display import display

button = widgets.Button(
    description="Approve",
    button_style="success"
)

display(button)

custom guardrails